# Traffic Demand Prediction - Improved Solution
OOF CV R2 Score: 0.972746 | Hackathon Score: 97.2746

In [1]:
import pandas as pdimport numpy as npfrom sklearn.model_selection import KFoldfrom sklearn.metrics import r2_scoreimport lightgbm as lgbimport xgboost as xgbfrom sklearn.ensemble import HistGradientBoostingRegressorimport jsonimport zipfileimport warningswarnings.filterwarnings('ignore')# =============================================================================# 1. Load Data# =============================================================================train_df = pd.read_csv('dataset/train.csv')test_df = pd.read_csv('dataset/test.csv')print(f"Train shape: {train_df.shape}")print(f"Test shape: {test_df.shape}")print(f"Target mean: {train_df['demand'].mean():.6f}, std: {train_df['demand'].std():.6f}")# =============================================================================# 2. Feature Engineering# =============================================================================def extract_time_features(df):    """Extract rich time-based features from timestamp."""    df = df.copy()        # Parse hour and minute    time_split = df['timestamp'].str.split(':', expand=True)    df['hour'] = time_split[0].astype(int)    df['minute'] = time_split[1].astype(int)        # Total minutes from midnight (continuous time)    df['total_minutes'] = df['hour'] * 60 + df['minute']        # 15-minute interval index (0-95)    df['time_slot'] = df['total_minutes'] // 15        # Cyclical encoding for hour (captures periodicity: 23:00 is close to 0:00)    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)        # Cyclical encoding for time_slot    df['slot_sin'] = np.sin(2 * np.pi * df['time_slot'] / 96)    df['slot_cos'] = np.cos(2 * np.pi * df['time_slot'] / 96)        # Time of day categories    df['is_morning_rush'] = ((df['hour'] >= 7) & (df['hour'] <= 10)).astype(int)    df['is_evening_rush'] = ((df['hour'] >= 16) & (df['hour'] <= 19)).astype(int)    df['is_night'] = ((df['hour'] >= 22) | (df['hour'] <= 5)).astype(int)    df['is_peak'] = ((df['hour'] >= 7) & (df['hour'] <= 13)).astype(int)        return dfdef encode_categoricals(df):    """Encode categorical features properly."""    df = df.copy()        # Binary encode LargeVehicles    df['LargeVehicles_enc'] = (df['LargeVehicles'] == 'Allowed').astype(int)        # Binary encode Landmarks    df['Landmarks_enc'] = (df['Landmarks'] == 'Yes').astype(int)        # Ordinal encode RoadType (based on demand analysis: Residential < Street < Highway)    road_map = {'Residential': 0, 'Street': 1, 'Highway': 2}    df['RoadType_enc'] = df['RoadType'].map(road_map)    # Fill NaN with mode (Residential)    df['RoadType_enc'] = df['RoadType_enc'].fillna(0)        # Also keep indicators for Highway and Street (strong signals)    df['is_highway'] = (df['RoadType'] == 'Highway').astype(int)    df['is_street'] = (df['RoadType'] == 'Street').astype(int)        # Weather encoding    weather_map = {'Sunny': 0, 'Rainy': 1, 'Foggy': 2, 'Snowy': 3}    df['Weather_enc'] = df['Weather'].map(weather_map)    df['Weather_enc'] = df['Weather_enc'].fillna(-1)  # flag for missing        # Weather one-hot (since there are only 4 categories)    for w in ['Sunny', 'Rainy', 'Foggy', 'Snowy']:        df[f'weather_{w.lower()}'] = (df['Weather'] == w).astype(int)        return dfdef build_geohash_features(train, test, target_col='demand'):    """Build geohash-based target encoding features using train data only."""        # Global mean for smoothing    global_mean = train[target_col].mean()    global_median = train[target_col].median()        # Geohash-level statistics from training data    geo_stats = train.groupby('geohash')[target_col].agg([        'mean', 'median', 'std', 'count', 'min', 'max'    ]).reset_index()    geo_stats.columns = ['geohash', 'geo_demand_mean', 'geo_demand_median',                          'geo_demand_std', 'geo_count', 'geo_demand_min', 'geo_demand_max']        # Smoothed target encoding (Bayesian mean with prior)    m = 10  # smoothing factor    geo_stats['geo_target_enc'] = (        (geo_stats['geo_count'] * geo_stats['geo_demand_mean'] + m * global_mean) /        (geo_stats['geo_count'] + m)    )        # Demand range per geohash    geo_stats['geo_demand_range'] = geo_stats['geo_demand_max'] - geo_stats['geo_demand_min']        # Fill std NaN (single-obs geohashes) with 0    geo_stats['geo_demand_std'] = geo_stats['geo_demand_std'].fillna(0)        # Merge into train and test    train = train.merge(geo_stats, on='geohash', how='left')    test = test.merge(geo_stats, on='geohash', how='left')        # Fill any test geohashes not in train with global stats    for col in ['geo_demand_mean', 'geo_demand_median', 'geo_target_enc']:        test[col] = test[col].fillna(global_mean)    test['geo_demand_std'] = test['geo_demand_std'].fillna(0)    test['geo_count'] = test['geo_count'].fillna(0)    test['geo_demand_min'] = test['geo_demand_min'].fillna(global_mean)    test['geo_demand_max'] = test['geo_demand_max'].fillna(global_mean)    test['geo_demand_range'] = test['geo_demand_range'].fillna(0)        # Geohash-hour interaction stats (how does demand vary by hour for each location?)    geo_hour_stats = train.copy()    geo_hour_stats['hour'] = geo_hour_stats['timestamp'].str.split(':').str[0].astype(int)    geo_hour_agg = geo_hour_stats.groupby(['geohash', 'hour'])[target_col].agg(['mean', 'count']).reset_index()    geo_hour_agg.columns = ['geohash', 'hour', 'geo_hour_demand_mean', 'geo_hour_count']        # Smoothed geo-hour target encoding    geo_hour_agg['geo_hour_target_enc'] = (        (geo_hour_agg['geo_hour_count'] * geo_hour_agg['geo_hour_demand_mean'] + m * global_mean) /        (geo_hour_agg['geo_hour_count'] + m)    )        # Merge    train['hour_temp'] = train['timestamp'].str.split(':').str[0].astype(int)    test['hour_temp'] = test['timestamp'].str.split(':').str[0].astype(int)        train = train.merge(geo_hour_agg[['geohash', 'hour', 'geo_hour_target_enc']],                         left_on=['geohash', 'hour_temp'], right_on=['geohash', 'hour'], how='left')    test = test.merge(geo_hour_agg[['geohash', 'hour', 'geo_hour_target_enc']],                       left_on=['geohash', 'hour_temp'], right_on=['geohash', 'hour'], how='left')        train['geo_hour_target_enc'] = train['geo_hour_target_enc'].fillna(train['geo_target_enc'])    test['geo_hour_target_enc'] = test['geo_hour_target_enc'].fillna(test['geo_target_enc'])        # Drop temp columns    train = train.drop(columns=['hour_temp', 'hour_y'], errors='ignore')    test = test.drop(columns=['hour_temp', 'hour_y'], errors='ignore')        # Rename if needed    if 'hour_x' in train.columns:        train = train.rename(columns={'hour_x': 'hour'})    if 'hour_x' in test.columns:        test = test.rename(columns={'hour_x': 'hour'})        # Geohash-RoadType interaction    geo_road = train.groupby(['geohash', 'RoadType'])[target_col].mean().reset_index()    geo_road.columns = ['geohash', 'RoadType', 'geo_road_demand']    train = train.merge(geo_road, on=['geohash', 'RoadType'], how='left')    test = test.merge(geo_road, on=['geohash', 'RoadType'], how='left')    test['geo_road_demand'] = test['geo_road_demand'].fillna(global_mean)        return train, testdef add_interaction_features(df):    """Add meaningful interaction features."""    df = df.copy()        # Road capacity proxy: NumberofLanes * is_highway    df['lanes_highway'] = df['NumberofLanes'] * df['is_highway']    df['lanes_street'] = df['NumberofLanes'] * df['is_street']        # High-capacity indicator    df['high_capacity'] = ((df['NumberofLanes'] >= 4) | (df['is_highway'] == 1)).astype(int)        # Temperature interactions    df['temp_is_highway'] = df['Temperature'] * df['is_highway']        # Geohash demand * hour interaction    df['geo_mean_x_hour'] = df['geo_demand_mean'] * df['hour_sin']        # Lane * large vehicles interaction    df['lanes_large_vehicles'] = df['NumberofLanes'] * df['LargeVehicles_enc']        # Ratio features    df['geo_mean_vs_median'] = df['geo_demand_mean'] / (df['geo_demand_median'] + 1e-8)        return dfdef full_pipeline(train_df, test_df):    """Full feature engineering pipeline."""    train = train_df.copy()    test = test_df.copy()        # Time features    train = extract_time_features(train)    test = extract_time_features(test)        # Categorical encoding    train = encode_categoricals(train)    test = encode_categoricals(test)        # Geohash target encoding    train, test = build_geohash_features(train, test)        # Interaction features    train = add_interaction_features(train)    test = add_interaction_features(test)        # Drop original columns not needed for modeling    drop_cols = ['timestamp', 'geohash', 'RoadType', 'LargeVehicles', 'Landmarks',                  'Weather', 'Index']        if 'demand' in train.columns:        y = train['demand']        train = train.drop(columns=['demand'] + [c for c in drop_cols if c in train.columns])    else:        y = None        train = train.drop(columns=[c for c in drop_cols if c in train.columns])        test_index = test_df['Index']    test = test.drop(columns=[c for c in drop_cols if c in test.columns])        return train, y, test, test_index# =============================================================================# 3. Build Features# =============================================================================X_train, y_train, X_test, test_index = full_pipeline(train_df, test_df)# Ensure feature alignmentcommon_cols = [c for c in X_train.columns if c in X_test.columns]X_train = X_train[common_cols]X_test = X_test[common_cols]print(f"\nFeatures: {len(common_cols)}")print(f"Feature names: {common_cols}")# Fill remaining NaN in Temperature with medianX_train['Temperature'] = X_train['Temperature'].fillna(X_train['Temperature'].median())X_test['Temperature'] = X_test['Temperature'].fillna(X_train['Temperature'].median())# =============================================================================# 4. Model Training with K-Fold CV + Ensemble# =============================================================================N_FOLDS = 5kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)# LightGBM parameters (tuned)lgb_params = {    'objective': 'regression',    'metric': 'rmse',    'boosting_type': 'gbdt',    'learning_rate': 0.03,    'num_leaves': 127,    'max_depth': -1,    'min_child_samples': 20,    'feature_fraction': 0.8,    'bagging_fraction': 0.8,    'bagging_freq': 5,    'reg_alpha': 0.1,    'reg_lambda': 1.0,    'n_estimators': 2000,    'verbose': -1,    'random_state': 42,}# XGBoost parameters (tuned)xgb_params = {    'objective': 'reg:squarederror',    'eval_metric': 'rmse',    'learning_rate': 0.03,    'max_depth': 8,    'min_child_weight': 10,    'subsample': 0.8,    'colsample_bytree': 0.8,    'reg_alpha': 0.1,    'reg_lambda': 1.0,    'n_estimators': 2000,    'random_state': 42,    'tree_method': 'hist',    'verbosity': 0,}oof_lgb = np.zeros(len(X_train))oof_xgb = np.zeros(len(X_train))test_lgb = np.zeros(len(X_test))test_xgb = np.zeros(len(X_test))print(f"\n{'='*60}")print(f"Training with {N_FOLDS}-Fold Cross Validation")print(f"{'='*60}")for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):    print(f"\n--- Fold {fold+1}/{N_FOLDS} ---")        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]        # LightGBM    lgb_model = lgb.LGBMRegressor(**lgb_params)    lgb_model.fit(        X_tr, y_tr,        eval_set=[(X_val, y_val)],        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)]    )    oof_lgb[val_idx] = lgb_model.predict(X_val)    test_lgb += lgb_model.predict(X_test) / N_FOLDS        lgb_r2 = r2_score(y_val, oof_lgb[val_idx])    print(f"  LGB R2: {lgb_r2:.6f} (score: {max(0, 100*lgb_r2):.4f})")        # XGBoost    xgb_model = xgb.XGBRegressor(**xgb_params)    xgb_model.fit(        X_tr, y_tr,        eval_set=[(X_val, y_val)],        verbose=False,    )    oof_xgb[val_idx] = xgb_model.predict(X_val)    test_xgb += xgb_model.predict(X_test) / N_FOLDS        xgb_r2 = r2_score(y_val, oof_xgb[val_idx])    print(f"  XGB R2: {xgb_r2:.6f} (score: {max(0, 100*xgb_r2):.4f})")# Overall OOF scoreslgb_oof_r2 = r2_score(y_train, oof_lgb)xgb_oof_r2 = r2_score(y_train, oof_xgb)print(f"\n{'='*60}")print(f"Overall OOF R2 - LGB: {lgb_oof_r2:.6f} (score: {max(0, 100*lgb_oof_r2):.4f})")print(f"Overall OOF R2 - XGB: {xgb_oof_r2:.6f} (score: {max(0, 100*xgb_oof_r2):.4f})")# =============================================================================# 5. Find Optimal Ensemble Weights# =============================================================================best_r2 = -1best_w = 0.5for w in np.arange(0.0, 1.01, 0.05):    blend = w * oof_lgb + (1 - w) * oof_xgb    r2 = r2_score(y_train, blend)    if r2 > best_r2:        best_r2 = r2        best_w = wprint(f"\nBest ensemble weight (LGB): {best_w:.2f}")print(f"Best ensemble OOF R2: {best_r2:.6f} (score: {max(0, 100*best_r2):.4f})")# Final blended predictionstest_preds = best_w * test_lgb + (1 - best_w) * test_xgb# Clip predictions to valid rangetest_preds = np.clip(test_preds, 0, 1)# =============================================================================# 6. Create Submission# =============================================================================sub_df = pd.DataFrame({    'Index': test_index,    'demand': test_preds})sub_df.to_csv('predictions.csv', index=False)print(f"\npredictions.csv created with {len(sub_df)} rows")# =============================================================================# 7. Feature Importance (from last fold LGB model)# =============================================================================importance = pd.DataFrame({    'feature': common_cols,    'importance': lgb_model.feature_importances_}).sort_values('importance', ascending=False)print(f"\nTop 15 Feature Importances:")print(importance.head(15).to_string(index=False))# =============================================================================# 8. Update Approach Document# =============================================================================approach_text = f"""Traffic Demand Prediction - Improved Approach1. Feature Engineering:   - Timestamp Parsing: Extracted hour, minute, total_minutes, and 15-min time_slot from timestamp.   - Cyclical Time Encoding: Applied sine/cosine encoding to hour and time_slot to capture periodicity.   - Time-of-Day Indicators: Created rush hour, night, and peak indicators.   - Geohash Target Encoding: Computed smoothed (Bayesian) mean, median, std, min, max,      range per geohash using training data only. Applied Laplace smoothing (m=10) to prevent      overfitting on low-count geohashes.   - Geohash-Hour Interaction: Target-encoded demand per (geohash, hour) pair for fine-grained      temporal-spatial patterns.   - Geohash-RoadType Interaction: Captured how demand varies by road type within each location.   - Categorical Encoding: Binary-encoded LargeVehicles, Landmarks. Ordinal-encoded RoadType      (Residential < Street < Highway). One-hot encoded Weather.   - Interaction Features: NumberofLanes Ã— is_highway, high_capacity indicator,      Temperature Ã— is_highway, geo_mean Ã— hour_sin, etc.   - Missing Value Handling: Filled Temperature NaNs with median. Used -1 flag for missing      Weather. Filled missing RoadType as Residential (mode).2. Modeling:   - Ensemble of LightGBM and XGBoost with {N_FOLDS}-fold cross validation.   - LightGBM: GBDT with lr=0.03, 127 leaves, 2000 trees, early stopping (50 rounds).   - XGBoost: Histogram-based with lr=0.03, depth=8, 2000 trees.   - Optimal blending weight found via grid search on OOF predictions.   - Best ensemble weight (LGB): {best_w:.2f}   - OOF CV R2 Score: {best_r2:.6f} (Hackathon Score: {max(0, 100*best_r2):.4f})   - Predictions clipped to [0, 1] range.3. Tools Used:   - Python 3   - pandas, numpy for data manipulation   - LightGBM, XGBoost for gradient boosting models   - scikit-learn for cross-validation and R2 metric"""with open('approach.txt', 'w') as f:    f.write(approach_text.strip())# =============================================================================# 9. Create Jupyter Notebook# =============================================================================# Read the current Python source to embed in notebookwith open('solution_v2.py', 'r') as f:    source_code = f.read()# Split into logical chunks for notebook cellsnotebook_content = {    "cells": [        {            "cell_type": "markdown",            "metadata": {},            "source": [                "# Traffic Demand Prediction - Improved Solution\n",                f"OOF CV R2 Score: {best_r2:.6f} | Hackathon Score: {max(0, 100*best_r2):.4f}"            ]        },        {            "cell_type": "code",            "execution_count": 1,            "metadata": {},            "outputs": [],            "source": source_code.split('\n')        }    ],    "metadata": {        "kernelspec": {            "display_name": "Python 3",            "language": "python",            "name": "python3"        },        "language_info": {            "name": "python",            "version": "3.8.0"        }    },    "nbformat": 4,    "nbformat_minor": 4}with open('solution.ipynb', 'w') as f:    json.dump(notebook_content, f, indent=1)# =============================================================================# 10. Create Zip Archive# =============================================================================with zipfile.ZipFile('submission.zip', 'w') as zipf:    zipf.write('predictions.csv')    zipf.write('approach.txt')    zipf.write('solution.ipynb')print("\nAll files created successfully!")print("- predictions.csv")print("- approach.txt")print("- solution.ipynb")print("- submission.zip")